# Shipping Management — Mock Data Generator

**Safe to run multiple times.** Each run clears existing rows (TRUNCATE) before inserting, so counts never exceed the numbers defined in the CONFIG cell.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import random
from datetime import datetime, timedelta, date

import pandas as pd
from faker import Faker
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

In [3]:
# ── CONFIG — number of rows get inserted ───────────────
DB_USER = "root"
DB_PASS = "11111"
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "shipping_management"

NUM_CUSTOMERS       = 5001
NUM_SERVICE_AREAS   = 101
NUM_EMPLOYEES       = 1001
NUM_WAREHOUSES      = 51
NUM_VENDORS         = 201
NUM_VEHICLES        = 501
NUM_MAINTENANCE     = 3001
NUM_FUEL_PURCHASES  = 5001
NUM_SHIPMENTS       = 50001
NUM_ROUTES          = 3001
NUM_CLAIMS          = 1001
NUM_TICKETS         = 1501
NUM_PACKAGES        = 100001
NUM_ROUTE_STOPS     = 30000
NUM_INVENTORY       = 100000
NUM_INVOICES        = 50001

In [5]:
# ── Database connection ───────────────────────────────────────────────────────
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True
)

try:
    with engine.connect() as conn:
        print("✅ Database connection successful!")
except Exception as e:
    raise SystemExit(f"❌ Connection failed: {e}")

✅ Database connection successful!


In [7]:
# ── Truncate all tables ─────────────────────────────────────────
# Order matters — child tables first, then parents.
TRUNCATE_ORDER = [
    "inventory",
    "route_stops",
    "invoices",
    "insurance_claims",
    "customer_service_tickets",
    "packages",
    "delivery_routes",
    "shipments",
    "fuel_purchases",
    "maintenance_records",
    "vehicles",
    "vendors",
    "warehouses",
    "employees",
    "service_area_zip_codes",
    "service_areas",
    "customers",
]

with engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    for table in TRUNCATE_ORDER:
        conn.execute(text(f"TRUNCATE TABLE `{table}`"))
        print(f"  Truncated {table}")
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))

print("\n✅ All tables cleared.")

  Truncated inventory
  Truncated route_stops
  Truncated invoices
  Truncated insurance_claims
  Truncated customer_service_tickets
  Truncated packages
  Truncated delivery_routes
  Truncated shipments
  Truncated fuel_purchases
  Truncated maintenance_records
  Truncated vehicles
  Truncated vendors
  Truncated warehouses
  Truncated employees
  Truncated service_area_zip_codes
  Truncated service_areas
  Truncated customers

✅ All tables cleared.


In [9]:
# ── Faker setup ───────────────────────────────────────────────────────────────
fake = Faker()
Faker.seed(42)          # reproducible data
random.seed(42)

In [11]:
# ── 1. customers ──────────────────────────────────────────────────────────────
print("Generating customers...")

used_emails  = set()
used_phones  = set()

def unique_email():
    while True:
        v = fake.email()
        if v not in used_emails:
            used_emails.add(v)
            return v

def unique_phone():
    while True:
        v = fake.phone_number()
        if v not in used_phones:
            used_phones.add(v)
            return v

customers = [{
    "customer_id":              i,
    "name":                     fake.name(),
    "email":                    unique_email(),
    "phone":                    unique_phone(),
    "address_line1":            fake.street_address(),
    "address_line2":            fake.secondary_address(),
    "city":                     fake.city(),
    "state":                    fake.state(),
    "postal_code":              fake.postcode(),
    "created_at":               fake.date_time_between(start_date="-5y", end_date="now"),
    "customer_type":            random.choice(["individual", "business"]),
    "preferred_contact_method": random.choice(["email", "sms"]),
    "loyalty_points":           random.randint(0, 1000),
} for i in range(NUM_CUSTOMERS)]

df_customers = pd.DataFrame(customers)
print(f"  {len(df_customers):,} rows ready")

Generating customers...
  5,001 rows ready


In [13]:
# ── 2. service_areas ──────────────────────────────────────────────────────────
print("Generating service_areas...")

service_areas = [{
    "area_id":                       i,
    "name":                          f"{fake.city()} Area",
    "description":                   fake.sentence(nb_words=8)[:255],
    "base_delivery_fee":             round(random.uniform(5, 50), 2),
    "estimated_transit_time_hours":  random.randint(1, 48),
    "service_level":                 random.choice(["standard", "expedited", "same-day"]),
} for i in range(NUM_SERVICE_AREAS)]

df_service_areas = pd.DataFrame(service_areas)
print(f"  {len(df_service_areas):,} rows ready")

Generating service_areas...
  101 rows ready


In [15]:
# ── 3. service_area_zip_codes ─────────────────────────────────────────────────
print("Generating service_area_zip_codes...")

used_zips = set()
zip_codes = []

for area in service_areas:
    num_zips = random.randint(5, 15)
    added = 0
    attempts = 0
    while added < num_zips and attempts < 10_000:
        z = fake.postcode()
        attempts += 1
        if z not in used_zips:
            used_zips.add(z)
            zip_codes.append({"area_id": area["area_id"], "zip_code": z})
            added += 1

df_zip_codes = pd.DataFrame(zip_codes)
print(f"  {len(df_zip_codes):,} rows ready")

Generating service_area_zip_codes...
  1,011 rows ready


In [17]:
# ── 4. employees ──────────────────────────────────────────────────────────────
print("Generating employees...")

used_emp_emails   = set()
used_emp_phones   = set()
used_licenses     = set()

def unique_emp_email():
    while True:
        v = fake.email()
        if v not in used_emp_emails:
            used_emp_emails.add(v)
            return v

def unique_emp_phone():
    while True:
        v = fake.phone_number()
        if v not in used_emp_phones:
            used_emp_phones.add(v)
            return v

def unique_license():
    while True:
        v = fake.bothify(text="??######")
        if v not in used_licenses:
            used_licenses.add(v)
            return v

JOB_TITLES  = ["Driver", "Manager", "Dispatcher", "Clerk", "Technician"]
DEPARTMENTS = ["Operations", "HR", "IT", "Logistics"]

employees = []
for i in range(NUM_EMPLOYEES):
    # managers only assigned once there are at least 10 employees
    if i > 10 and random.random() < 0.7:
        manager_id = random.randint(0, i - 1)
    else:
        manager_id = None

    employees.append({
        "employee_id":          i,
        "first_name":           fake.first_name(),
        "last_name":            fake.last_name(),
        "email":                unique_emp_email(),
        "phone":                unique_emp_phone(),
        "hire_date":            fake.date_between(start_date="-5y", end_date="today"),
        "job_title":            random.choice(JOB_TITLES),
        "department":           random.choice(DEPARTMENTS),
        "manager_id":           manager_id,
        "monthly_salary":       round(random.uniform(2000, 8000), 2),
        "address":              fake.address().replace("\n", ", ")[:255],
        "emergency_contact":    fake.name(),
        "driver_license_number": unique_license(),
    })

df_employees = pd.DataFrame(employees)
print(f"  {len(df_employees):,} rows ready")

Generating employees...
  1,001 rows ready


In [19]:
# ── 5. warehouses ─────────────────────────────────────────────────────────────
print("Generating warehouses...")

used_wh_addresses = set()

def unique_wh_address():
    while True:
        v = fake.address().replace("\n", ", ")[:255]
        if v not in used_wh_addresses:
            used_wh_addresses.add(v)
            return v

warehouses = []
for i in range(NUM_WAREHOUSES):
    start_h = random.randint(0, 11)
    end_h   = random.randint(12, 23)
    warehouses.append({
        "warehouse_id":              i,
        "name":                      f"Warehouse {i}",
        "address":                   unique_wh_address(),
        "city":                      fake.city(),
        "state":                     fake.state(),
        "postal_code":               fake.postcode(),
        "created_at":                fake.date_between(start_date="-10y", end_date="-1y"),
        "updated_at":                fake.date_between(start_date="-1y",  end_date="today"),
        # square_footage is SMALLINT (max 32767) — cap at 30000
        "square_footage":            random.randint(1000, 30000),
        "dock_doors_count":          random.randint(2, 20),
        # Store as HH:MM:SS strings so SQLAlchemy maps them correctly
        "start_time":                f"{start_h:02d}:00:00",
        "end_time":                  f"{end_h:02d}:00:00",
        # manager_id must be a valid employee_id (0 to NUM_EMPLOYEES-1)
        "manager_id":                random.randint(0, NUM_EMPLOYEES - 1),
        "temperature_controlled":    random.choice([True, False]),
        "hazardous_material_allowed": random.choice([True, False]),
    })

df_warehouses = pd.DataFrame(warehouses)
print(f"  {len(df_warehouses):,} rows ready")

Generating warehouses...
  51 rows ready


In [21]:
# ── 6. vendors ────────────────────────────────────────────────────────────────
print("Generating vendors...")

vendors = [{
    "vendor_id":              i,
    "name":                   fake.company(),
    "contact_person":         fake.name(),
    "email":                  fake.company_email(),
    "phone":                  fake.phone_number(),
    "service_type":           random.choice(["fuel", "maintenance", "insurance"]),
    "contract_start_date":    fake.date_between(start_date="-5y", end_date="-1y"),
    "contract_end_date":      fake.date_between(start_date="today", end_date="+2y"),
    "preferred_vendor_status": random.choice(["preferred", "non-preferred"]),
} for i in range(NUM_VENDORS)]

df_vendors = pd.DataFrame(vendors)
print(f"  {len(df_vendors):,} rows ready")

Generating vendors...
  201 rows ready


In [23]:
# ── 7. vehicles ───────────────────────────────────────────────────────────────
print("Generating vehicles...")

used_plates = set()

def unique_plate():
    while True:
        v = fake.bothify(text="??###??")
        if v not in used_plates:
            used_plates.add(v)
            return v

vehicles = []
for i in range(NUM_VEHICLES):
    year        = random.randint(2010, 2024)
    acq_date    = fake.date_between(start_date=date(year, 1, 1), end_date=date(year, 12, 31))
    last_insp   = fake.date_between(start_date=acq_date, end_date="today")
    next_insp   = fake.date_between(start_date="today",  end_date="+1y")
    vehicles.append({
        "vehicle_id":           i,
        "license_plate":        unique_plate(),
        "make":                 fake.company()[:50],
        "model":                fake.word(),
        "year":                 year,
        "vehicle_type":         random.choice(["van", "box-truck"]),
        # SMALLINT max is 32767 — keep within range
        "max_weight_capacity":  random.randint(1000, 10000),
        "max_volume_capacity":  random.randint(200,  1500),
        "current_status":       random.choice(["available", "in-maintenance", "out-of-service"]),
        "acquisition_date":     acq_date,
        "last_inspection_date": last_insp,
        "next_inspection_date": next_insp,
        "fuel_type":            random.choice(["gasoline", "diesel", "electric", "hybrid"]),
        "average_mpg":          random.randint(5, 25),
        "current_mileage":      random.randint(10000, 300000),
    })

df_vehicles = pd.DataFrame(vehicles)
print(f"  {len(df_vehicles):,} rows ready")

Generating vehicles...
  501 rows ready


In [25]:
# ── 8. maintenance_records ────────────────────────────────────────────────────
print("Generating maintenance_records...")

maintenance_records = []
for i in range(NUM_MAINTENANCE):
    veh             = random.choice(vehicles)
    maint_date      = fake.date_between(start_date=veh["acquisition_date"], end_date="today")
    maintenance_records.append({
        "maintenance_id":    i,
        "vehicle_id":        veh["vehicle_id"],
        "vendor_id":         random.randint(0, NUM_VENDORS - 1),
        "maintenance_type":  random.choice(["oil change", "brake inspection", "tire rotation", "engine check"]),
        "description":       fake.sentence(nb_words=6)[:255],
        "date_performed":    maint_date,
        "mileage_at_service": random.randint(5000, veh["current_mileage"]),
        "cost":              round(random.uniform(50, 1000), 2),
        "part_used":         fake.word(),
        "next_service_date": fake.date_between(start_date=maint_date, end_date="+6m"),
        "technician_note":   fake.sentence(nb_words=8)[:255],
    })

df_maintenance = pd.DataFrame(maintenance_records)
print(f"  {len(df_maintenance):,} rows ready")

Generating maintenance_records...
  3,001 rows ready


In [27]:
# ── 9. fuel_purchases ─────────────────────────────────────────────────────────
print("Generating fuel_purchases...")

used_receipts = set()

def unique_receipt():
    while True:
        v = fake.bothify(text="RCPT-####-???")
        if v not in used_receipts:
            used_receipts.add(v)
            return v

# Driver IDs = employees whose job_title is 'Driver'
driver_ids = [e["employee_id"] for e in employees if e["job_title"] == "Driver"]

fuel_purchases = []
for i in range(NUM_FUEL_PURCHASES):
    gallons = round(random.uniform(5, 50), 2)
    price   = round(random.uniform(2.5, 6.0), 2)
    fuel_purchases.append({
        "purchase_id":      i,
        "vehicle_id":       random.randint(0, NUM_VEHICLES - 1),
        "vendor_id":        random.randint(0, NUM_VENDORS - 1),
        "purchase_date":    fake.date_time_between(start_date="-2y", end_date="now"),
        "gallons":          gallons,
        "price_per_gallon": price,
        "total_cost":       round(gallons * price, 2),
        "odometer_reading": round(random.uniform(5000, 300000), 2),
        "payment_method":   random.choice(["cash", "credit-cards", "debit-cards", "fuel-cards"]),
        "receipt_number":   unique_receipt(),
        "driver_id":        random.choice(driver_ids) if driver_ids else None,
    })

df_fuel = pd.DataFrame(fuel_purchases)
print(f"  {len(df_fuel):,} rows ready")

Generating fuel_purchases...
  5,001 rows ready


In [29]:
# ── 10. shipments ─────────────────────────────────────────────────────────────
print("Generating shipments...")

dispatcher_ids = [e["employee_id"] for e in employees if e["job_title"] == "Dispatcher"]

shipments = []
for i in range(NUM_SHIPMENTS):
    pickup          = fake.date_between(start_date="-2y", end_date="-1d")
    est_delivery    = pickup + timedelta(days=random.randint(1, 5))
    actual_delivery = est_delivery + timedelta(days=random.choice([-1, 0, 1, 2]))

    shipments.append({
        "shipment_id":              i,
        "customer_id":              random.randint(0, NUM_CUSTOMERS - 1),
        "origin_warehouse_id":      random.randint(0, NUM_WAREHOUSES - 1),
        "destination_address":      fake.street_address(),
        "destination_city":         fake.city(),
        "destination_state":        fake.state(),
        "destination_postal_code":  fake.postcode(),
        "service_area_id":          random.randint(0, NUM_SERVICE_AREAS - 1),
        "pickup_date":              pickup,
        "estimated_delivery_date":  datetime.combine(est_delivery, datetime.min.time()),
        "actual_delivery_date":     datetime.combine(actual_delivery, datetime.min.time()),
        "status":                   random.choice(["pending", "in-transit", "delivered", "delayed", "cancelled"]),
        "priority_level":           random.choice(["low", "medium", "high", "urgent"]),
        "special_instructions":     fake.sentence(nb_words=6)[:255],
        "dispatcher_id":            random.choice(dispatcher_ids) if dispatcher_ids else random.randint(0, NUM_EMPLOYEES - 1),
        "billing_notes":            fake.sentence(nb_words=5)[:255],
    })

df_shipments = pd.DataFrame(shipments)
print(f"  {len(df_shipments):,} rows ready")

Generating shipments...
  50,001 rows ready


In [30]:
# ── 11. delivery_routes ───────────────────────────────────────────────────────
print("Generating delivery_routes...")

routes = []
for i in range(NUM_ROUTES):
    start_at = fake.date_time_between(start_date="-2y", end_date="-1d")
    end_at   = start_at + timedelta(hours=random.randint(2, 10))
    routes.append({
        "route_id":              i,
        "driver_id":             random.choice(driver_ids) if driver_ids else random.randint(0, NUM_EMPLOYEES - 1),
        "vehicle_id":            random.randint(0, NUM_VEHICLES - 1),
        "start_warehouse_id":    random.randint(0, NUM_WAREHOUSES - 1),
        "start_at":              start_at,
        "end_at":                end_at,
        "status":                random.choice(["scheduled", "in-progress", "paused", "completed", "cancelled"]),
        "total_stops_planned":   random.randint(3, 20),
        "total_stops_completed": random.randint(0, 20),
        "miles_estimated":       round(random.uniform(10, 300), 2),
        "miles_actual":          round(random.uniform(10, 350), 2),
        "fuel_consumed_gallons": round(random.uniform(2, 40), 2),
        # CHECK constraint: 1 <= route_score_by_driver <= 5
        "route_score_by_driver": round(random.uniform(1, 5), 2),
    })

df_routes = pd.DataFrame(routes)
print(f"  {len(df_routes):,} rows ready")

Generating delivery_routes...
  3,001 rows ready


In [33]:
# ── 12. insurance_claims ──────────────────────────────────────────────────────
print("Generating insurance_claims...")

adjuster_ids = [e["employee_id"] for e in employees]

insurance_claims = []
for i in range(NUM_CLAIMS):
    claim_date = fake.date_between(start_date="-1y", end_date="-1d")
    status     = random.choice(["pending", "approved", "denied"])
    settlement_date   = claim_date + timedelta(days=random.randint(5, 30)) if status != "pending" else None
    settlement_amount = round(random.uniform(50, 500), 2)                  if settlement_date else None

    insurance_claims.append({
        "claim_id":            i,
        "shipment_id":         random.randint(0, NUM_SHIPMENTS - 1),
        "customer_id":         random.randint(0, NUM_CUSTOMERS - 1),
        "claim_date":          claim_date,
        "claim_amount":        round(random.uniform(50, 1000), 2),
        "damage_description":  fake.sentence(nb_words=5)[:100],
        "claim_status":        status,
        "settlement_date":     settlement_date,
        "settlement_amount":   settlement_amount,
        "adjuster_id":         random.choice(adjuster_ids),
        "investigation_notes": fake.sentence(nb_words=6)[:100],
    })

df_claims = pd.DataFrame(insurance_claims)
print(f"  {len(df_claims):,} rows ready")

Generating insurance_claims...
  1,001 rows ready


In [35]:
# ── 13. customer_service_tickets ──────────────────────────────────────────────
print("Generating customer_service_tickets...")

tickets = []
for i in range(NUM_TICKETS):
    # CHECK constraint: 1 <= customer_satisfaction_rating <= 5
    rating = round(random.uniform(1.0, 5.0), 1)
    tickets.append({
        "ticket_id":                   i,
        "customer_id":                 random.randint(0, NUM_CUSTOMERS - 1),
        "shipment_id":                 random.randint(0, NUM_SHIPMENTS - 1),
        "employee_id":                 random.randint(0, NUM_EMPLOYEES - 1),
        "ticket_date":                 fake.date_between(start_date="-1y", end_date="-1d"),
        "issue_type":                  random.choice(["delivery_delay", "damaged_package", "missing_item", "billing_issue"]),
        "description":                 fake.sentence(nb_words=6)[:255],
        "resolution":                  random.choice(["Refund issued", "Resent package", "Escalated to billing", "Investigation closed"]),
        "status":                      random.choice(["open", "in-progress", "resolved", "escalated"]),
        "priority":                    random.choice(["low", "medium", "high", "urgent"]),
        "resolution_time_minutes":     random.randint(10, 1000),
        "customer_satisfaction_rating": rating,
    })

df_tickets = pd.DataFrame(tickets)
print(f"  {len(df_tickets):,} rows ready")

Generating customer_service_tickets...
  1,501 rows ready


In [37]:
# ── 14. packages ──────────────────────────────────────────────────────────────
print("Generating packages...")

# Use the row index to guarantee uniqueness — no collision possible ever.
# Format: TRK-00000000 (8-digit zero-padded index) + 2 random letters for variety.
def make_tracking(i):
    suffix = fake.bothify(text="??")
    return f"TRK-{i:08d}-{suffix}"

packages = []
for i in range(NUM_PACKAGES):
    packages.append({
        "package_id":           i,
        "shipment_id":          random.randint(0, NUM_SHIPMENTS - 1),
        "description":          fake.sentence(nb_words=4)[:255],
        "weight":               round(random.uniform(0.5, 75), 2),
        "height":               round(random.uniform(5, 100), 2),
        "width":                round(random.uniform(5, 100), 2),
        "length":               round(random.uniform(5, 100), 2),
        "package_type":         random.choice(["Box", "Envelope", "Crate", "Tube"]),
        "tracking_number":      make_tracking(i),
        "is_fragile":           random.choice([True, False]),
        "is_perishable":        random.choice([True, False]),
        "requires_signature":   random.choice([True, False]),
        "declared_value":       round(random.uniform(10, 2000), 2),
        "customs_info":         fake.sentence(nb_words=3)[:255],
        "handling_instructions": fake.sentence(nb_words=5)[:255],
    })

df_packages = pd.DataFrame(packages)
print(f"  {len(df_packages):,} rows ready")

Generating packages...
  100,001 rows ready


In [39]:
# ── 15. route_stops ───────────────────────────────────────────────────────────
print("Generating route_stops...")

route_stops = []
for i in range(NUM_ROUTE_STOPS):
    est_arrival  = fake.date_time_between(start_date="-1y", end_date="now")
    act_arrival  = est_arrival  + timedelta(minutes=random.randint(0, 60))
    departure    = act_arrival  + timedelta(minutes=random.randint(5, 30))
    route_stops.append({
        "stop_id":               i,
        "route_id":              random.randint(0, NUM_ROUTES - 1),
        "sequence_number":       random.randint(1, 10),
        "shipment_id":           random.randint(0, NUM_SHIPMENTS - 1),
        "estimated_arrival":     est_arrival,
        "actual_arrival":        act_arrival,
        "departure_time":        departure,
        # stop_duration_minutes is TINYINT (max 127)
        "stop_duration_minutes": random.randint(5, 60),
        "stop_status":           random.choice(["pending", "completed", "skipped", "failed"]),
        "delivery_notes":        fake.sentence(nb_words=4)[:255],
        "recipient_signature":   fake.name(),
        "proof_of_delivery_url": fake.url(),
    })

df_route_stops = pd.DataFrame(route_stops)
print(f"  {len(df_route_stops):,} rows ready")

Generating route_stops...
  30,000 rows ready


In [41]:
# ── 16. inventory ─────────────────────────────────────────────────────────────
print("Generating inventory...")

used_loc_codes = set()

def unique_loc_code():
    while True:
        v = "LOC-" + fake.bothify(text="??????????")
        if v not in used_loc_codes:
            used_loc_codes.add(v)
            return v

inventory = []
for i in range(NUM_INVENTORY):
    date_received = fake.date_time_between(start_date="-1y", end_date="now")
    date_shipped  = date_received + timedelta(days=random.randint(1, 5)) if random.random() > 0.3 else None
    inventory.append({
        "inventory_id":  i,
        "warehouse_id":  random.randint(0, NUM_WAREHOUSES - 1),
        "package_id":    random.randint(0, NUM_PACKAGES - 1),
        "location_code": unique_loc_code(),
        "date_received": date_received,
        "date_shipped":  date_shipped,
        "current_status": random.choice(["received", "staged", "loaded", "delivered"]),
        "handled_by":    random.randint(0, NUM_EMPLOYEES - 1) if random.random() > 0.1 else None,
        "temperature_log": round(random.uniform(-5.0, 35.0), 2),
    })

df_inventory = pd.DataFrame(inventory)
print(f"  {len(df_inventory):,} rows ready")

Generating inventory...
  100,000 rows ready


In [43]:
# ── 17. invoices ──────────────────────────────────────────────────────────────
print("Generating invoices...")

invoices = []
for i in range(NUM_INVOICES):
    issue_date       = fake.date_time_between(start_date="-1y", end_date="now")
    due_date         = issue_date + timedelta(days=15)
    total_amount     = round(random.uniform(50, 500), 2)
    tax_amount       = round(total_amount * 0.1, 2)
    discount_amount  = round(random.uniform(0, 20), 2)
    paid             = random.random() > 0.1
    amount_paid      = total_amount - discount_amount if paid else 0.0
    payment_date     = issue_date + timedelta(days=random.randint(1, 20)) if paid else None
    payment_status   = "paid" if paid else random.choice(["pending", "overdue", "refunded"])

    invoices.append({
        "invoice_id":       i,
        "shipment_id":      random.randint(0, NUM_SHIPMENTS - 1),
        "customer_id":      random.randint(0, NUM_CUSTOMERS - 1),
        "issue_date":       issue_date,
        "due_date":         due_date,
        "total_amount":     total_amount,
        "tax_amount":       tax_amount,
        "discount_amount":  discount_amount,
        "amount_paid":      amount_paid,
        "payment_date":     payment_date,
        "payment_method":   random.choice(["credit_card", "bank_transfer", "paypal", "cash"]),
        "payment_status":   payment_status,
        "late_fee_amount":  round(random.uniform(0, 10), 2) if payment_status == "overdue" else 0.0,
    })

df_invoices = pd.DataFrame(invoices)
print(f"  {len(df_invoices):,} rows ready")

Generating invoices...
  50,001 rows ready


In [45]:
# ── Insert all tables ─────────────────────────────────────────────────────────
# Ordered so every FK parent is inserted before its children.
INSERT_ORDER = [
    (df_customers,    "customers"),
    (df_service_areas, "service_areas"),
    (df_zip_codes,    "service_area_zip_codes"),
    (df_employees,    "employees"),
    (df_warehouses,   "warehouses"),
    (df_vendors,      "vendors"),
    (df_vehicles,     "vehicles"),
    (df_maintenance,  "maintenance_records"),
    (df_fuel,         "fuel_purchases"),
    (df_shipments,    "shipments"),
    (df_routes,       "delivery_routes"),
    (df_claims,       "insurance_claims"),
    (df_tickets,      "customer_service_tickets"),
    (df_packages,     "packages"),
    (df_route_stops,  "route_stops"),
    (df_inventory,    "inventory"),
    (df_invoices,     "invoices"),
]

CHUNK_SIZE = 5000   # rows per batch — keeps memory stable for large tables

try:
    with engine.begin() as conn:
        for df, table_name in INSERT_ORDER:
            total = len(df)
            for start in range(0, total, CHUNK_SIZE):
                chunk = df.iloc[start : start + CHUNK_SIZE]
                chunk.to_sql(name=table_name, con=conn, if_exists="append", index=False)
            print(f"  ✅  {table_name:<30} {total:>8,} rows inserted")

    print("\n✅ All tables inserted successfully.")

except SQLAlchemyError as e:
    print(f"\n❌ Transaction failed — all changes rolled back.\n   Error: {e}")

  ✅  customers                         5,001 rows inserted
  ✅  service_areas                       101 rows inserted
  ✅  service_area_zip_codes            1,011 rows inserted
  ✅  employees                         1,001 rows inserted
  ✅  warehouses                           51 rows inserted
  ✅  vendors                             201 rows inserted
  ✅  vehicles                            501 rows inserted
  ✅  maintenance_records               3,001 rows inserted
  ✅  fuel_purchases                    5,001 rows inserted
  ✅  shipments                        50,001 rows inserted
  ✅  delivery_routes                   3,001 rows inserted
  ✅  insurance_claims                  1,001 rows inserted
  ✅  customer_service_tickets          1,501 rows inserted
  ✅  packages                        100,001 rows inserted
  ✅  route_stops                      30,000 rows inserted
  ✅  inventory                       100,000 rows inserted
  ✅  invoices                         50,001 rows insert